<a href="https://colab.research.google.com/github/Spinomk1/Programacion-sistemas-base-1/blob/main/Espinosa_Cristopher_PSSB_I_Colab_U1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Programación de Sistemas de Base I - Inspección de la Estructura de un Compilador y Procesamiento de Lenguajes
**Asignatura:** Programación de Sistemas de Base I (8.º Semestre)  
**Unidad I:** Introducción a la Compilación  
**Tiempo estimado:** 3 horas (Asíncrono / Guiado)  
**Repositorio Base / Entrega:** Integración con Moodle / GitHub

---


## 1. Objetivos de Aprendizaje
1. Identificar de forma tangible la diferencia entre ejecución compilada e interpretada utilizando las herramientas nativas del entorno (`gcc`, `python3`, `dis`, `ast`).
2. Explorar y visualizar la fase de Análisis Léxico y Sintáctico mediante la inspección del Árbol de Sintaxis Abstracta (AST) y la generación de Bytecode.
3. Comprender el rol de la Tabla de Símbolos en el seguimiento de identificadores durante el proceso de traducción.

---


## 2. Conexión Teórica (NotebookLM)
**Antes de continuar:** Accede al espacio de **NotebookLM del Curso** y realiza las siguientes preguntas de verificación:

- *¿Cuáles son las fases de la etapa de Análisis (Front-End) de un compilador?*  
**RESPUESTA:**

- *¿Qué diferencia a un Lexema de un Token?*  
**RESPUESTA:**

- *¿Qué función cumple la Tabla de Símbolos durante la compilación?*  
**RESPUESTA:**


---

## 3. Sección Práctica 1: Pipeline de Compilación vs. Interpretación (CLI / Bash)
En esta sección analizaremos cómo el sistema operativo y el entorno procesan un lenguaje compilado (C) frente a uno interpretado/híbrido (Python).

### [CÓDIGO 1.1] Inspección de Herramientas del Sistema
```bash
!echo "=== COMPILADOR C DE LINUX (GCC) ==="
!gcc --version | head -n 1
!echo ""
!echo "=== INTÉRPRETE DE PYTHON ==="
!python3 --version

In [ ]:
!gcc --version | head -n 1

gcc (Ubuntu 11.4.0-1ubuntu1~22.04.3) 11.4.0


### [CÓDIGO 1.2] Creación, Compilación y Análisis de Binario en C
```c
%%writefile hola_compilador.c
#include <stdio.h>
#define inputMsj "Escriba un numero entero: "
#define msj "La suma de %d + %d es: %d\n"

int sumar(int x, int y){
  return x + y;
}

int main() {
    int a = 5;
    int b;
    
    printf(inputMsj);
    scanf("%d",&b);
    int suma = sumar(a, b);
    printf(msj, a, b, suma);
    return 0;
}

In [ ]:
%%writefile hola_compilador.c
# include <stdio.h>
# define inputMsj "Escriba un numero entero: "
# define msj "La suma de %d + %d es: %d\n"

int sumar(int x, int y){
  return x + y;
}

int main() {
    int a = 5;
    int b;

    printf(inputMsj);
    scanf("%d",&b);
    int suma = sumar(a, b);
    printf(msj, a, b, suma);
    return 0;
}

Overwriting hola_compilador.c


## El Preprocesador (Antes del ensamblador)
Esta etapa expande las macros (como `#define`) e incluye los archivos de cabecera (como `#include <stdio.h>`).
```bash
!gcc -E hola_compilador.c -o hola_compilador.i
!echo === CÓDIGO PREPROCESADO ===
!cat hola_compilador.i | tail -n 20


In [ ]:
!cat hola_compilador.i | tail -n 20


# 2 "hola_compilador.c" 2




# 5 "hola_compilador.c"
int sumar (int x, int y){
  return x + y;
}

int main(){
  int a = 5;
  int b;
   printf("Escriba un número entero: ");
   scanf ("%d", &b);
   int suma = sumar(a, b);
   printf("LA suma de %d + %d es: %d\n", a, b, suma);
   return 0;
}


## Representación Intermedia (IR) y Optimizaciones
GCC utiliza una representación intermedia llamada GIMPLE antes de generar el ensamblador. Puedes ver cómo cambia el código antes y después de que el optimizador trabaje.

```bash
!gcc -O2 -fdump-tree-gimple hola_compilador.c -c
!echo === REPRESENTACIÓN INTERMEDIA (GIMPLE) ===
!cat hola_compilador.c.*gimple

In [ ]:
!gcc -O2 -fdump-tree-gimple hola_compilador.c -c
!echo "=== REPRESENTACIÓN INTERMEDIA (GIMPLE) ==="
!cat hola_compilador.c.*gimple

hola_compilador.c: In function ‘main’:
hola_compilador.c:14:5: warning: ignoring return value of ‘scanf’ declared with attribute ‘warn_unused_result’ []8;;https://gcc.gnu.org/onlinedocs/gcc/Warning-Options.html#index-Wunused-result-Wunused-result]8;;]
   14 |     scanf("%d",&b);
      |     ^~~~~~~~~~~~~~
=== REPRESENTACIÓN INTERMEDIA (GIMPLE) ===
int sumar (int x, int y)
{
  int D.2555;

  D.2555 = x + y;
  return D.2555;
}


int main ()
{
  int D.2557;

  {
    int a;
    int b;
    int suma;

    try
      {
        a = 5;
        printf ("Escriba un numero entero: ");
        scanf ("%d", &b);
        b.0_1 = b;
        suma = sumar (a, b.0_1);
        b.1_2 = b;
        printf ("La suma de %d + %d es: %d\n", a, b.1_2, suma);
        D.2557 = 0;
        return D.2557;
      }
    finally
      {
        b = {CLOBBER};
      }
  }
  D.2557 = 0;
  return D.2557;
}


__attribute__((artificial, gnu_inline, always_inline))
__attribute__((nonnull (1), format (printf, 1, 2)))
int prin

## Compilación generando código ensamblador intermediate (.s)
 El compilador traduce el código **sin optimizaciones** línea por línea directamente a la memoria RAM (la pila o stack)

```bash
!gcc -S hola_compilador.c -o hola_compilador.s
!echo "=== CÓDIGO ENSAMBLADOR GENERADO (SÍNTESIS) ==="
!cat hola_compilador.s | head -n 25

In [ ]:
!gcc -S hola_compilador.c -o hola_compilador.s
!echo "===Código ensanmbaldor generado"
!cat hola_compilador.s | head -n 25

===Código ensanmbaldor generado
	.file	"hola_compilador.c"
	.text
	.globl	sumar
	.type	sumar, @function
sumar:
.LFB0:
	.cfi_startproc
	endbr64
	pushq	%rbp
	.cfi_def_cfa_offset 16
	.cfi_offset 6, -16
	movq	%rsp, %rbp
	.cfi_def_cfa_register 6
	movl	%edi, -4(%rbp)
	movl	%esi, -8(%rbp)
	movl	-4(%rbp), %edx
	movl	-8(%rbp), %eax
	addl	%edx, %eax
	popq	%rbp
	.cfi_def_cfa 7, 8
	ret
	.cfi_endproc
.LFE0:
	.size	sumar, .-sumar
	.section	.rodata


Para producir una traducción del código **optimizada** usa

```bash
!gcc -O3 -S hola_compilador.c -o hola_conmpilador_opt.s
!echo "=== CÓDIGO ENSAMBLADOR OPTIMIZADO GENERADO (SÍNTESIS) ==="
!cat hola_compilador_opt.s | head -n 25

In [ ]:
!gcc -O3 -S hola_compilador.c -o hola_compilador_opt.s
!echo "===Código ensamblador optimizado generado==="
!cat hola_compilador_opt.s | head -n 25

hola_compilador.c: In function ‘main’:
hola_compilador.c:14:5: warning: ignoring return value of ‘scanf’ declared with attribute ‘warn_unused_result’ []8;;https://gcc.gnu.org/onlinedocs/gcc/Warning-Options.html#index-Wunused-result-Wunused-result]8;;]
   14 |     scanf("%d",&b);
      |     ^~~~~~~~~~~~~~
===Código ensamblador optimizado generado===
	.file	"hola_compilador.c"
	.text
	.p2align 4
	.globl	sumar
	.type	sumar, @function
sumar:
.LFB23:
	.cfi_startproc
	endbr64
	leal	(%rdi,%rsi), %eax
	ret
	.cfi_endproc
.LFE23:
	.size	sumar, .-sumar
	.section	.rodata.str1.1,"aMS",@progbits,1
.LC0:
	.string	"Escriba un numero entero: "
.LC1:
	.string	"%d"
.LC2:
	.string	"La suma de %d + %d es: %d\n"
	.section	.text.startup,"ax",@progbits
	.p2align 4
	.globl	main
	.type	main, @function


## Superando limitaciones de la arquitectura fija (X86_64)
La máquina virtual de Colab corre sobre procesadores Intel o AMD de 64 bits.  
**Impacto:** El código ensamblador generado (.s) siempre estará en sintaxis AT&T (por defecto en Linux) y para arquitectura x86_64. Para ver código ensamblador con sintaxis de Intel (más parecida a la de Windows), agregar el parámetro ***-masm=intel*** a la orden de GCC:

```bash
!gcc -S -masm=intel hola_compilador.c -o hola_compilador_8664.s
!echo "=== CÓDIGO ENSAMBLADOR x86_64 GENERADO (SÍNTESIS) ==="
!cat hola_compilador_8664.s | head -n 25


In [ ]:
!gcc -S -masm=intel hola_compilador.c -o hola_compilador_8664.s
!echo "===código ensamblador  x86_64 generado==="
!cat hola_compilador_8664.s | head -n 25

===código ensamblador  x86_64 generado===
	.file	"hola_compilador.c"
	.intel_syntax noprefix
	.text
	.globl	sumar
	.type	sumar, @function
sumar:
.LFB0:
	.cfi_startproc
	endbr64
	push	rbp
	.cfi_def_cfa_offset 16
	.cfi_offset 6, -16
	mov	rbp, rsp
	.cfi_def_cfa_register 6
	mov	DWORD PTR -4[rbp], edi
	mov	DWORD PTR -8[rbp], esi
	mov	edx, DWORD PTR -4[rbp]
	mov	eax, DWORD PTR -8[rbp]
	add	eax, edx
	pop	rbp
	.cfi_def_cfa 7, 8
	ret
	.cfi_endproc
.LFE0:
	.size	sumar, .-sumar


## Compilación a código objeto
```bash
# Generación del código objeto (antes de enlazar) y conteo líneas
!gcc -c hola_compilador.c -o hola_compilador.o
!objdump -d hola_compilador.o | wc -l

In [ ]:
!gcc -c hola_compilador.c -o hola_compilador.o
!objdump -d hola_compilador.o | wc -l

60


## Compilación a código máquina ejecutable

```bash
# Generación del código ejecutable final (después de enlazar) y conteo líneas
!gcc hola_compilador.c -o hola_compilador
!objdump -d hola_compilador | wc -l


In [ ]:
!gcc hola_compilador.c -o hola_compilador
!objdump -d hola_compilador | wc -l

194


## El "Monstruo Completo" (Enlace Estático)  
Por defecto, GCC usa ***enlace dinámico*** usando una estructura de datos tabular, la tabla **PLT** (*Procedure Linkage Table*) ya que el código usa `printf`, el programa necesita conectarse con la biblioteca estándar de C (`libc.so`). El enlazador no copia el código de `printf` dentro del archivo para no duplicar espacio en el disco; en su lugar, crea un "trampolín" o acceso directo en una sección llamada `<printf@plt>`.

Cada vez que se invoca a `printf`, realmente salta a este bloque plt, el cual averigua en qué parte de la memoria RAM del sistema operativo se encuentra la verdadera función `printf` y redirige el control allí en tiempo de ejecución.

Para ver cómo el enlazador inyecta literalmente miles y miles de líneas copiando físicamente el código de `printf` y todas sus dependencias dentro del ejecutable, se debe compilar de forma estática:

```bash
# Compilación estática
!gcc -static hola_compilador.c -o hola_estatico

# Conteo de líneas de código ensamblador
!objdump -d hola_estatico | wc -l


In [ ]:
!gcc -static hola_compilador.c -o hola_estatico

cc1: fatal error: hola_compilador.c: No such file or directory
compilation terminated.


## Ejecución del código
```bash
!echo ""
!echo "=== EJECUCIÓN DEL PROGRAMA OBJETO ==="
!./hola_compilador


In [ ]:
!echo ""
!echo "=== EJECUCIÓN DEL PROGRAMA OBJETO ==="
!./hola_compilador


=== EJECUCIÓN DEL PROGRAMA OBJETO ===
/bin/bash: line 1: ./hola_compilador: No such file or directory


## 4. Sección Práctica 2: Revelando el Front-End (Análisis Léxico y AST)
Para comprender cómo el compilador "desarma" el código fuente en tokens y estructuras jerárquicas, inspeccionaremos el compilador de Python a nivel interno.

### [CÓDIGO 2.1] Inspección del Analizador Léxico (Tokenizador)

```python
import token
import tokenize
from io import BytesIO

# Código fuente de prueba en formato cadena
codigo_fuente = "suma = a + 10"

# Tokenización del código fuente
tokens = tokenize.tokenize(BytesIO(codigo_fuente.encode("utf-8")).readline)

print(
    f"{'LINEA/COL':<12} | {'TIPO DE TOKEN':<20} | {'VALOR (LEXEMA)':<15}"
)
print("-" * 55)
for tok in tokens:
    if tok.type in (
        tokenize.ENCODING,
        tokenize.ENDMARKER,
        tokenize.NL,
        tokenize.NEWLINE,
    ):
        continue
    nombre_token = token.tok_name[tok.type]
    posicion = f"{tok.start[0]}:{tok.start[1]}"
    print(f"{posicion:<12} | {nombre_token:<20} | {tok.string:<15}")


### [CÓDIGO 2.2] Construcción del Árbol de Sintaxis Abstracta (AST)

```python
import ast

# Generar y visualizar la estructura sintáctica
arbol = ast.parse("suma = a + 10")
print("=== ÁRBOLES DE SINTAXIS ABSTRACTA (REPRESENTACIÓN EN TEXTO) ===")
print(ast.dump(arbol, indent=4))


## 5. Sección Práctica 3: Inspección de la Tabla de Símbolos y Bytecode
En esta sección simularemos la interacción con la Tabla de Símbolos y observaremos la generación de código intermedio.

### [CÓDIGO 3.1] Simulación de una Tabla de Símbolos básica en Python
```python
class TablaDeSimbolos:

    def __init__(self):
        self.simbolos = {}

    def insertar(self, nombre, tipo, valor=None, ambito="global"):
        if nombre in self.simbolos:
            print(f"[ERROR LÉXICO/SINTÁCTICO] Identificador '{nombre}' ya declarado.")
        else:
            self.simbolos[nombre] = {
                "tipo": tipo,
                "valor": valor,
                "ambito": ambito,
            }
            print(f"[TABLA DE SÍMBOLOS] Insertado: {nombre} ({tipo})")

    def buscar(self, nombre):
        return self.simbolos.get(nombre, None)

    def mostrar(self):
        print("\n=== CONTENIDO DE LA TABLA DE SÍMBOLOS ===")
        print(f"{'NOMBRE':<12} | {'TIPO':<10} | {'ÁMBITO':<10} | {'VALOR':<10}")
        print("-" * 50)
        for nombre, datos in self.simbolos.items():
            print(
                f"{nombre:<12} | {datos['tipo']:<10} | {datos['ambito']:<10} | {str(datos['valor']):<10}"
            )


# Prueba de la Tabla de Símbolos
ts = TablaDeSimbolos()
ts.insertar("a", "ENTERO", 5)
ts.insertar("b", "ENTERO", 10)
ts.insertar("suma", "ENTERO", 15)
ts.mostrar()
```


### [CÓDIGO 3.2] Desensamblado a Código Intermedio / Bytecode (Máquina de Pila)
```python
import dis

def calcular():
    a = 5
    b = 10
    suma = a + b
    return suma


print("=== BYTECODE GENERADO PARA LA MÁQUINA VIRTUAL DE PYTHON ===")
dis.dis(calcular)
```

## 6. Desafío Asíncrono / Entregable de la Unidad I (Avance del Producto Integrador)

### Contexto del Proyecto Integrador Autónomo:
Durante el semestre, cada equipo concebirá, diseñará e implementará un **Analizador Léxico y Sintáctico (Compiler Front-End)** para un lenguaje original propuesto por el propio equipo.

Ejemplos de proyectos de semestres anteriores:
- **DSL para Configuración de Robots / Drones:** Lenguaje de comandos simples (`FORWARD 10`, `ROTATE 90`).
- **Lenguaje de Consultas para Grafos/Tablas:** Alternativa simplificada a SQL (`SELECT age FROM users WHERE status == 1`).
- **Mini-Lenguaje Matemático / Scripting:** Soporte para vectores, matrices o evaluación de expresiones lógicas/aritméticas.
- **Lenguaje Formato/Marcado Personalizado:** Generador de reportes en HTML/Markdown a partir de una sintaxis limpia.

---

### Instrucciones del Desafío U1 (Fase 0: Definición e Inspección Inicial):

#### Parte A: Documento de Especificación del Lenguaje (RFC del Equipo)
Crea un archivo `ESPECIFICACION.docx` que contenga:
1. **Nombre del Lenguaje Original y Propósito:** ¿Qué problema resuelve o a quién va dirigido?
2. **Ejemplo de Código Fuente Válido:** Muestra un fragmento de código de al menos 10-15 líneas escrito en tu nuevo lenguaje.
3. **Catálogo Preliminar de Tokens:** Lista las palabras reservadas, identificadores, constantes (enteras, flotantes, cadenas) y operadores que usará tu lenguaje.

#### Parte B: Prototipo de Inspección en Código (Colab Execution)
Utilizando las celdas mágicas `%%writefile` (en Python o C):
1. Crea un archivo con una cadena que contenga tu ejemplo de código fuente original.
2. Implementa una función de prueba preliminar (o utiliza la librería `tokenize` de Python como simulador) que tome la cadena de tu lenguaje y la desglose en una lista de componentes léxicos.
3. Muestra una estructura en código que sirva como **Tabla de Símbolos inicial** donde se registren las variables declaradas en tu lenguaje.

---

### Criterios Mínimos que Debe Cumplir Cualquier Lenguaje Propuesto (Checklist de Viabilidad):
Para que la propuesta sea aprobada por el catedrático, el lenguaje debe cumplir con:
- [ ] Poseer al menos **3 tipos de tokens bien diferenciados** (ej. Palabras Reservadas, Identificadores, Literales).
- [ ] Incluir soporte para **expresiones aritméticas o lógicas** anidadas.
- [ ] Incluir al menos una **estructura de control de flujo** (ej. `if/else`, `while`, `repeat`) o una **estructura de bloques** (funciones, comandos).
- [ ] Definir un mecanismo explícito de asignación o declaración de datos.

---

### Autoevaluación Muestreada (Comprobación Tipo Gradiance)
Responde las siguientes preguntas analizando la sintaxis de tu nuevo lenguaje y valida tus razonamientos en NotebookLM:

1. **Pregunta 1 (Conflictos Léxicos):** Al revisar las palabras reservadas y los identificadores de tu lenguaje, ¿existe alguna regla léxica que pudiera causar ambigüedad (por ejemplo, que una palabra reservada coincida con el patrón de un identificador de usuario)? ¿Cómo la resolverá tu analizador?
2. **Pregunta 2 (Estructura de la Tabla de Símbolos):** De los elementos de tu lenguaje original, ¿cuáles atributos (tipo, valor, ámbito, dirección de memoria) necesitará almacenar tu Tabla de Símbolos cuando se procese una declaración?

---

### Formato de Entrega / Portafolio de Evidencias
1. Guarda este cuaderno con todas las salidas ejecutadas (`Archivo` -> `Guardar una copia en GitHub` / `PEREZ JUAN PSSB I Colab U1.ipynb`).
2. Guarda el documento `ESPECIFICACION.docx` con la especificación de tu lenguaje en tu repositorio personal (no repositorios grupales) de GitHub.
3. Registra en **Moodle** el enlace del cuaderno ejecutable en Colab y el de tu repositorio conteniendo la especificación formal del proyecto.

## Sección de trabajo no dirigido
Prototipo de Inspección en Código (Colab Execution)
Utilizando las celdas mágicas `%%writefile` (en Python o C):
1. Crea un archivo con una cadena que contenga tu ejemplo de código fuente original.

In [ ]:
%%writefile my_code.c

FUNCTION FuncionEjemplo(param1: INT, param2: STRING) RETURNS BOOL:
    IF param1 > 10 THEN
        PRINT("Número grande")
        RETURN TRUE
    ELSE
        PRINT("Número pequeño")
        RETURN FALSE
    END IF
END FUNCTION

VAR myVar = 5
CALL FuncionEjemplo(myVar, "Hola")

Writing my_code.c


2. Implementa una función de prueba preliminar (o utiliza la librería `tokenize` de Python como simulador) que tome la cadena de tu lenguaje y la desglose en una lista de componentes léxicos.

In [ ]:
import re

def simple_tokenizer(code_string):
    keywords = ["FUNCTION", "INT", "STRING", "RETURNS", "BOOL", "IF", "THEN", "ELSE", "END IF", "END FUNCTION", "VAR", "CALL", "PRINT", "RETURN"]
    operators = [":", "(", ")", ">", "="]
    literals = ['\"', '[0-9]+'] # String literals and integer literals
    delimiters = [" ", "\n", "\t"]

    tokens = []
    # Combine all patterns for splitting, ensuring multi-character operators/keywords are matched first
    # Use lookarounds to keep the delimiters/operators in the split result
    token_patterns = '|'.join(map(re.escape, sorted(keywords + operators, key=len, reverse=True)))
    split_pattern = f'({token_patterns}|\s+)'

    parts = re.split(split_pattern, code_string)

    for part in parts:
        if not part or part.isspace():
            continue
        if part in keywords:
            tokens.append((part, 'KEYWORD'))
        elif part in operators:
            tokens.append((part, 'OPERATOR'))
        elif re.fullmatch(r'"[^"\\]*(?:\\.[^"\\]*)*"', part): # Regex for string literals
            tokens.append((part, 'STRING_LITERAL'))
        elif re.fullmatch(r'\d+', part): # Regex for integer literals
            tokens.append((part, 'INTEGER_LITERAL'))
        elif re.fullmatch(r'[a-zA-Z_][a-zA-Z0-9_]*', part): # Regex for identifiers
            tokens.append((part, 'IDENTIFIER'))
        else:
            tokens.append((part, 'UNKNOWN'))

    return tokens

# Read the custom language code from the file
try:
    with open('my_code.c', 'r') as f:
        custom_code = f.read()
except FileNotFoundError:
    custom_code = "FUNCTION Example() RETURNS INT: RETURN 0 END FUNCTION"
    print("Warning: 'my_code.c' not found. Using a default example code.")

# Tokenize the custom code
token_list = simple_tokenizer(custom_code)

print(f"{'TOKEN':<25} | {'TYPE':<20}")
print("-" * 46)
for token, token_type in token_list:
    print(f"{token:<25} | {token_type:<20}")

TOKEN                     | TYPE                
----------------------------------------------
FUNCTION                  | KEYWORD             
FuncionEjemplo            | IDENTIFIER          
(                         | OPERATOR            
param1                    | IDENTIFIER          
:                         | OPERATOR            
INT                       | KEYWORD             
,                         | UNKNOWN             
param2                    | IDENTIFIER          
:                         | OPERATOR            
STRING                    | KEYWORD             
)                         | OPERATOR            
RETURNS                   | KEYWORD             
BOOL                      | KEYWORD             
:                         | OPERATOR            
IF                        | KEYWORD             
param1                    | IDENTIFIER          
>                         | OPERATOR            
10                        | INTEGER_LITERAL     
THEN                  

<>:13: SyntaxWarning: invalid escape sequence '\s'
<>:13: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_587/2313860780.py:13: SyntaxWarning: invalid escape sequence '\s'
  split_pattern = f'({token_patterns}|\s+)'


3. Muestra una estructura en código que sirva como **Tabla de Símbolos inicial** donde se registren las variables declaradas en tu lenguaje.

In [ ]:
import re

class SymbolTable:
    def __init__(self):
        self.symbols = {}
        self.current_scope = "global" # Para gestionar el ámbito actual

    def insert(self, name, symbol_type, category, value=None, scope=None, params=None):
        scope_to_use = scope if scope else self.current_scope

        # Crear un identificador único para el símbolo (nombre + ámbito)
        unique_id = f"{scope_to_use}::{name}"

        if unique_id in self.symbols:
            print(f"Error: '{name}' ya declarado en el ámbito '{scope_to_use}'. Saltando la inserción.")
            return

        self.symbols[unique_id] = {
            "name": name,
            "type": symbol_type,
            "category": category, # ej. 'variable', 'function', 'parameter'
            "value": value,
            "scope": scope_to_use,
            "parameters": params # Solo para funciones: lista de diccionarios [{'name': 'p1', 'type': 'INT'}]
        }

    def lookup(self, name, scope=None):
        scope_to_check = scope if scope else self.current_scope
        unique_id = f"{scope_to_check}::{name}"
        return self.symbols.get(unique_id)

    def display(self):
        print("\n=== CONTENIDO DE LA TABLA DE SÍMBOLOS ===")
        print(f"{'NOMBRE':<25} | {'TIPO':<10} | {'CATEGORÍA':<12} | {'VALOR':<10} | {'ÁMBITO':<15}")
        print("-" * 85)
        for unique_id, data in self.symbols.items():
            param_str = ""
            if data['category'] == 'function' and data['parameters']:
                param_str = "(" + ", ".join([f"{p['name']}:{p['type']}" for p in data['parameters']]) + ")"

            display_name = data['name'] + param_str
            print(
                f"{display_name:<25} | {data['type']:<10} | {data['category']:<12} | {str(data['value']):<10} | {data['scope']:<15}"
            )


# Crear una instancia de la Tabla de Símbolos
symbol_table = SymbolTable()

# Procesar la lista de tokens para popular la tabla de símbolos
index = 0
tokens = token_list # Usando la token_list de la celda anterior

while index < len(tokens):
    token_val, token_type = tokens[index]

    # Manejar Declaración de Función
    if token_val == "FUNCTION" and token_type == "KEYWORD":
        index += 1
        func_name = tokens[index][0]
        symbol_table.current_scope = func_name # Establecer el ámbito actual para los parámetros

        index += 1 # Avanzar más allá del nombre de la función

        # Saltar cualquier token UNKNOWN (como comas o partes de cadenas mal formadas) o espacios antes de '('
        while index < len(tokens) and (tokens[index][1] == 'UNKNOWN' or tokens[index][0].isspace()):
            index += 1

        if index >= len(tokens) or tokens[index][0] != '(': # Se espera '(' después del nombre de la función
            print(f"Error: Se esperaba '(' después del nombre de la función '{func_name}' en el token {index}. Encontrado: {tokens[index][0] if index < len(tokens) else 'EOF'}")
            index += 1 # Intentar avanzar para evitar un bucle infinito
            continue

        params = []
        index += 1 # Avanzar más allá de '('

        while index < len(tokens) and tokens[index][0] != ')':
            # Saltar cualquier token UNKNOWN o espacios
            while index < len(tokens) and (tokens[index][1] == 'UNKNOWN' or tokens[index][0].isspace()):
                index += 1

            if index >= len(tokens) or tokens[index][0] == ')': # Fin de los parámetros
                break

            param_name = tokens[index][0]
            index += 1

            # Saltar cualquier token UNKNOWN o espacios antes de ':'
            while index < len(tokens) and (tokens[index][1] == 'UNKNOWN' or tokens[index][0].isspace()):
                index += 1

            if index >= len(tokens) or tokens[index][0] != ':': # Se espera ':' después del nombre del parámetro
                print(f"Error: Se esperaba ':' después del nombre del parámetro '{param_name}' en la función '{func_name}' en el token {index}. Encontrado: {tokens[index][0] if index < len(tokens) else 'EOF'}")
                # Intentar recuperar saltando al siguiente token probable o ')'
                while index < len(tokens) and tokens[index][0] != ',' and tokens[index][0] != ')':
                    index += 1
                continue

            index += 1 # Avanzar más allá de ':'

            # Saltar cualquier token UNKNOWN o espacios antes del tipo de parámetro
            while index < len(tokens) and (tokens[index][1] == 'UNKNOWN' or tokens[index][0].isspace()):
                index += 1

            if index >= len(tokens):
                print(f"Error: Se esperaba un tipo de parámetro después de ':' para el parámetro '{param_name}' en la función '{func_name}' en el token {index}.")
                break

            param_type = tokens[index][0] # Asumiendo que el tipo es un solo token (KEYWORD o IDENTIFIER)

            params.append({"name": param_name, "type": param_type})
            symbol_table.insert(param_name, param_type, "parameter", scope=func_name)

            index += 1
            # El siguiente token debería ser ',' o ')'
            # Saltar cualquier token UNKNOWN o espacios
            while index < len(tokens) and (tokens[index][1] == 'UNKNOWN' or tokens[index][0].isspace()):
                index += 1

            if index < len(tokens) and tokens[index][0] == ',':
                index += 1 # Avanzar más allá de ','

        index += 1 # Avanzar más allá de ')' (fin de los parámetros)

        # Saltar cualquier token UNKNOWN o espacios antes de 'RETURNS'
        while index < len(tokens) and (tokens[index][1] == 'UNKNOWN' or tokens[index][0].isspace()):
            index += 1

        if index >= len(tokens) or tokens[index][0] != 'RETURNS':
            print(f"Error: Se esperaba 'RETURNS' después de los parámetros de la función '{func_name}' en el token {index}. Encontrado: {tokens[index][0] if index < len(tokens) else 'EOF'}")
            # Intentar recuperar saltando al siguiente token probable
            while index < len(tokens) and tokens[index][0] != ':':
                 index += 1
            if index < len(tokens) and tokens[index][0] == ':':
                index += 1 # Saltar ':' si sigue, asumiendo que lleva al cuerpo de la función
            continue

        index += 1 # Avanzar más allá de 'RETURNS'

        # Saltar cualquier token UNKNOWN o espacios antes del tipo de retorno
        while index < len(tokens) and (tokens[index][1] == 'UNKNOWN' or tokens[index][0].isspace()):
            index += 1

        if index >= len(tokens):
            print(f"Error: Se esperaba un tipo de retorno después de 'RETURNS' para la función '{func_name}' en el token {index}.")
            break

        return_type = tokens[index][0]

        # Insertar la función en la tabla de símbolos
        symbol_table.insert(func_name, return_type, "function", params=params, scope="global")
        symbol_table.current_scope = "global" # Reiniciar el ámbito después de la cabecera de la declaración de la función

    # Manejar Declaración de Variables
    elif token_val == "VAR" and token_type == "KEYWORD":
        index += 1

        # Saltar cualquier token UNKNOWN o espacios antes del nombre de la variable
        while index < len(tokens) and (tokens[index][1] == 'UNKNOWN' or tokens[index][0].isspace()):
            index += 1

        if index >= len(tokens) or tokens[index][1] != "IDENTIFIER":
            print(f"Error: Se esperaba un IDENTIFIER después de 'VAR' en el token {index}.")
            # Intentar recuperar
            while index < len(tokens) and tokens[index][0] != '=' and tokens[index][0] != ';':
                 index += 1
            if index < len(tokens) and tokens[index][0] == '=':
                index += 1 # Saltar '='
            continue

        var_name = tokens[index][0]
        index += 1

        # Saltar cualquier token UNKNOWN o espacios antes de '='
        while index < len(tokens) and (tokens[index][1] == 'UNKNOWN' or tokens[index][0].isspace()):
            index += 1

        if index < len(tokens) and tokens[index][0] == '=': # Se espera '=' después del nombre de la variable
            index += 1 # Avanzar más allá de '='

            # Saltar cualquier token UNKNOWN o espacios antes del valor
            while index < len(tokens) and (tokens[index][1] == 'UNKNOWN' or tokens[index][0].isspace()):
                index += 1

            if index >= len(tokens):
                print(f"Error: Se esperaba un valor después de '=' para la variable '{var_name}' en el token {index}.")
                break

            var_value = tokens[index][0]
            var_type = "UNKNOWN" # Inferir el tipo o extraerlo de una declaración explícita
            if tokens[index][1] == "INTEGER_LITERAL":
                var_type = "INT"
            elif tokens[index][1] == "STRING_LITERAL":
                var_type = "STRING"
            # Añadir otros tipos según sea necesario

            symbol_table.insert(var_name, var_type, "variable", value=var_value, scope="global")
        else:
            # Si no hay '=', podríamos inferir que es una declaración sin inicialización, o un error.
            # Para este ejemplo, si no hay '=', lo marcamos como error, ya que 'VAR' implica asignación.
            print(f"Error: Se esperaba '=' después del nombre de la variable '{var_name}' en el token {index}. Encontrado: {tokens[index][0] if index < len(tokens) else 'EOF'}")

    index += 1

symbol_table.display()


=== CONTENIDO DE LA TABLA DE SÍMBOLOS ===
NOMBRE                    | TIPO       | CATEGORÍA    | VALOR      | ÁMBITO         
-------------------------------------------------------------------------------------
param1                    | INT        | parameter    | None       | FuncionEjemplo 
param2                    | STRING     | parameter    | None       | FuncionEjemplo 
FuncionEjemplo(param1:INT, param2:STRING) | BOOL       | function     | None       | global         
myVar                     | INT        | variable     | 5          | global         


1. **Pregunta 1 (Conflictos Léxicos):** Al revisar las palabras reservadas y los identificadores de tu lenguaje, ¿existe alguna regla léxica que pudiera causar ambigüedad (por ejemplo, que una palabra reservada coincida con el patrón de un identificador de usuario)? ¿Cómo la resolverá tu analizador?

**RESPUESTA:**

Al revisar las palabras reservadas y el patrón de identificadores de nuestro lenguaje, existe una posible ambigüedad teórica: un usuario podría intentar usar una palabra reservada (ej. `IF`, `VAR`) como nombre para un identificador (variable o función).

Nuestro analizador léxico (el `simple_tokenizer` implementado) resuelve este conflicto mediante la **prioridad de coincidencia**. En la implementación, las palabras reservadas se verifican primero (`if part in keywords:`). Si una cadena coincide con una palabra reservada, se clasifica como `KEYWORD` y el proceso de búsqueda para identificarla como un `IDENTIFIER` no se ejecuta para esa cadena. Esto garantiza que las palabras reservadas siempre sean reconocidas como tales y no puedan ser utilizadas como identificadores por el usuario, eliminando así el conflicto léxico.

2. **Pregunta 2 (Estructura de la Tabla de Símbolos):** De los elementos de tu lenguaje original, ¿cuáles atributos (tipo, valor, ámbito, dirección de memoria) necesitará almacenar tu Tabla de Símbolos cuando se procese una declaración?

**RESPUESTA:**

Para los elementos de nuestro lenguaje original, la Tabla de Símbolos necesitará almacenar los siguientes atributos al procesar una declaración, ya sea de una variable o una función:

1.  **Nombre (Name):** El identificador del lexema (nombre de la variable, nombre de la función, nombre del parámetro). Esto es fundamental para la búsqueda y referencia.

2.  **Tipo (Type):** El tipo de dato asociado al símbolo (ej. `INT`, `STRING`, `BOOL`). Este atributo es crucial para el análisis semántico, permitiendo realizar la verificación de tipos y asegurando que las operaciones se apliquen a datos compatibles.

3.  **Categoría (Category):** Indicar si el símbolo es una `variable`, una `function` o un `parameter`. Esto ayuda a diferenciar cómo se debe interpretar y usar el símbolo dentro del contexto del programa.

4.  **Ámbito (Scope):** El contexto o región del código donde el símbolo es válido (ej. `global`, nombre de la función `FuncionEjemplo`). Esto es esencial para resolver conflictos de nombres y para la correcta visibilidad de los identificadores.

5.  **Valor (Value):** Para variables, este atributo podría almacenar el valor inicial asignado a la variable (ej. `5` para `myVar`). Aunque no siempre se usa en las primeras etapas de la compilación, es útil para la evaluación de constantes o para futuras etapas de optimización y ejecución.

6.  **Parámetros (Parameters):** Específico para las funciones. Una lista que describe los parámetros de la función, incluyendo su `nombre` y `tipo`. Esto es vital para verificar que las llamadas a funciones se realicen con el número y tipo correctos de argumentos.

7.  **Dirección de Memoria (Memory Address) / Offset:** Este atributo no es estrictamente necesario en las fases iniciales (léxico y sintáctico), pero se vuelve indispensable en la fase de generación de código. Almacenaría la ubicación en memoria (o un desplazamiento relativo) donde se asignará la variable o la dirección de entrada de una función. Nuestro prototipo actual no lo implementa, ya que está enfocado en las fases de análisis léxico y de tabla de símbolos, previas a la asignación de memoria.